# Algoritmo di Grover: studio sperimentale
## 1. Titolo e obiettivo del progetto
Studieremo amplificazione di ampiezza, overshooting e scaling con un'implementazione manuale in Python e Qiskit. Questo notebook costituisce la struttura iniziale per una futura relazione scientifica: in questa fase implementiamo e verifichiamo phase oracle e diffusion operator, oltre ad ambiente e strutture dati.

## 2. Descrizione sintetica dell'esperimento
Confronteremo la probabilità di trovare una soluzione marcata al variare delle iterazioni, del numero di qubit e del numero di soluzioni. Partiremo dalla simulazione ideale, aggiungeremo rumore e valuteremo eventualmente hardware IBM. Le funzioni high-level potranno servire in seguito come validazione.

Ogni futura esecuzione produrrà una riga di dati; i controlli di questa fase non produrranno risultati sperimentali.

## 3. Import e configurazione ambiente
Carichiamo solo le librerie necessarie alla struttura e alle verifiche dei due operatori e stampiamo le versioni per rendere tracciabile l'ambiente. Selezionare il kernel **Python (Grover)**, oppure l'interprete `.venv/Scripts/python.exe`.

Usiamo `AerSimulator` dal pacchetto separato `qiskit_aer`, compatibile con Qiskit 2.x; nelle prossime fasi useremo `transpile` e `backend.run`. Riferimento: [documentazione ufficiale AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html). Non importiamo ancora modelli di rumore, strumenti IBM o algoritmi preconfezionati.

In [1]:
import sys
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

LIBRARY_VERSIONS = {
    name: version(name)
    for name in ("qiskit", "qiskit-aer", "numpy", "pandas", "matplotlib")
}
print(f"Python: {sys.version.split()[0]}")
for name, installed_version in LIBRARY_VERSIONS.items():
    print(f"{name}: {installed_version}")
from qiskit.quantum_info import Operator, Statevector
from IPython.display import display

Python: 3.12.0
qiskit: 2.5.2
qiskit-aer: 0.17.2
numpy: 2.4.4
pandas: 3.0.2
matplotlib: 3.10.9


## 4. Parametri globali
Centralizziamo i valori iniziali e il seme casuale. Gli stati sono stringhe nell'ordine Qiskit `q[n-1]...q[0]`: il bit più a destra corrisponde al qubit 0. Partiamo da una soluzione; la tupla consentirà di rappresentarne più di una.

Il seme sarà passato anche al simulatore e al transpiler nei futuri esperimenti. Non rende riproducibili rumore fisico o tempi di esecuzione.

In [2]:
RANDOM_SEED = 42
SHOTS_DEFAULT = 1024
N_VALUES = (2, 3, 4, 5)  # Valori iniziali di n per il futuro scaling.
N_QUBITS_DEFAULT = N_VALUES[1]
MARKED_STATE_DEFAULT = "101"
MARKED_STATES_DEFAULT = (MARKED_STATE_DEFAULT,)
SEARCH_SPACE_SIZE = 2 ** N_QUBITS_DEFAULT
N_SOLUTIONS_DEFAULT = len(MARKED_STATES_DEFAULT)

rng = np.random.default_rng(RANDOM_SEED)


def validate_problem(n: int, marked_states: list[str] | tuple[str, ...]) -> None:
    """Controlla dimensione, alfabeto binario e unicità delle soluzioni."""
    if isinstance(n, bool) or not isinstance(n, int) or n < 1:
        raise ValueError("n deve essere un intero positivo.")
    if not isinstance(marked_states, (list, tuple)):
        raise ValueError("marked_states deve essere una lista o tupla di bitstring.")
    if not marked_states:
        raise ValueError("Occorre almeno una soluzione marcata.")
    if any(not isinstance(state, str) or len(state) != n
           or set(state) - {"0", "1"} for state in marked_states):
        raise ValueError("Ogni stato deve essere una stringa binaria di lunghezza n.")
    if len(set(marked_states)) != len(marked_states):
        raise ValueError("Gli stati marcati devono essere distinti.")


def make_ideal_backend(seed: int = RANDOM_SEED) -> AerSimulator:
    """Crea un simulatore ideale riproducibile senza eseguire circuiti."""
    return AerSimulator(seed_simulator=seed)


validate_problem(N_QUBITS_DEFAULT, MARKED_STATES_DEFAULT)

## 5. Costruzione dell'oracolo
Per uno stato marcato $O=I-2|w\rangle\langle w|$. L'ortonormalità della base dà
$O|x\rangle=(1-2\delta_{xw})|x\rangle$: solo l'ampiezza marcata cambia segno.
Per un insieme $W$ di stati distinti, $O_W=I-2\sum_{w\in W}|w\rangle\langle w|$.
Per ogni bitstring applichiamo X sui bit zero, una fase multi-controllata su
$|11\ldots1\rangle$, poi annulliamo gli X. Le riflessioni sui diversi stati marcati
si compongono senza interferire; i duplicati sono rifiutati perché annullerebbero il cambio di segno.

**Endianness:** le stringhe seguono l'ordine `q[n-1]...q[0]`: il bit a destra è il qubit 0.
`"101"` indica $q_2=1,q_1=0,q_0=1$ e l'indice `int("101", 2) = 5` nello Statevector.
`"001"` indica $q_2=0,q_1=0,q_0=1$, indice 1, non 4. Usiamo
`enumerate(reversed(state))` per associare caratteri e qubit. Nei diagrammi `q_0` è in alto.
Riferimento: [ordine dei bit in Qiskit](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering).

L'helper condiviso usa Z per n=1, CZ per n=2, e H–MCX–H sul target per n>=3.
Poiché $HXH=Z$, questa sequenza cambia segno solo quando tutti i qubit valgono 1.
Non usiamo ancille o GroverOperator.

In [3]:
def apply_multi_controlled_z(circuit: QuantumCircuit) -> None:
    """Applica in-place la fase -1 su |11...1> a tutti i qubit."""
    n = circuit.num_qubits
    if n < 1:
        raise ValueError("La Z multi-controllata richiede almeno un qubit.")
    if n == 1:
        circuit.z(0)
    elif n == 2:
        circuit.cz(0, 1)
    else:
        target = n - 1
        circuit.h(target)
        circuit.mcx(list(range(target)), target)
        circuit.h(target)


def build_phase_oracle(
    n_qubits: int, marked_states: list[str] | tuple[str, ...]
) -> QuantumCircuit:
    """Costruisce O_W; stringhe in ordine q[n-1]...q[0]."""
    validate_problem(n_qubits, marked_states)
    circuit = QuantumCircuit(n_qubits, name="O")
    for state in marked_states:
        zero_qubits = [q for q, bit in enumerate(reversed(state)) if bit == "0"]
        for q in zero_qubits:
            circuit.x(q)
        apply_multi_controlled_z(circuit)
        for q in reversed(zero_qubits):
            circuit.x(q)
    return circuit


def build_oracle(n: int, marked_states: list[str] | tuple[str, ...]) -> QuantumCircuit:
    """Alias compatibile con la firma già prevista nel notebook."""
    return build_phase_oracle(n, marked_states)

## 6. Costruzione del diffusore
Lo stato uniforme è $|s\rangle=H^{\otimes n}|0\ldots0\rangle$ e
$D=2|s\rangle\langle s|-I$. Scrivendo $|\psi\rangle=\alpha|s\rangle+|\psi_\perp\rangle$,
si ha $D|\psi\rangle=\alpha|s\rangle-|\psi_\perp\rangle$: la componente lungo lo stato
uniforme rimane invariata, quella ortogonale cambia segno. È una riflessione rispetto a $|s\rangle$.
Per $N=2^n$, ampiezze $a_x$ e media $\bar a=N^{-1}\sum_x a_x$, segue
$a'_x=2\bar a-a_x$ (**inversion about the mean**), anche per ampiezze complesse.

La sequenza H, X, multi-controlled Z, X, H realizza
$H^{\otimes n}(I-2|0\ldots0\rangle\langle0\ldots0|)H^{\otimes n}=-D$.
Includiamo `global_phase = np.pi` per ottenere esattamente $D$ e confrontare direttamente le ampiezze.
Una fase globale $e^{i\phi}$ non cambia le probabilità: $|e^{i\phi}a_x|^2=|a_x|^2$.
Le fasi relative tra componenti, invece, possono cambiare l'interferenza.

In [4]:
def build_diffuser(n_qubits: int) -> QuantumCircuit:
    """Costruisce D = 2|s><s| - I, inclusa la fase globale."""
    if isinstance(n_qubits, bool) or not isinstance(n_qubits, int) or n_qubits < 1:
        raise ValueError("n_qubits deve essere un intero positivo.")
    circuit = QuantumCircuit(n_qubits, name="D")
    circuit.h(range(n_qubits))
    circuit.x(range(n_qubits))
    apply_multi_controlled_z(circuit)
    circuit.x(range(n_qubits))
    circuit.h(range(n_qubits))
    circuit.global_phase = np.pi
    return circuit

### 6.1 Circuiti a tre qubit: stato marcato |101⟩
I diagrammi testuali mostrano le porte senza dipendenze grafiche aggiuntive.

In [5]:
oracle_example = build_phase_oracle(3, ["101"])
diffuser_example = build_diffuser(3)
print("Phase oracle, stato marcato |101>:")
print(oracle_example.draw(output="text"))
print("Diffusore (fase globale pi inclusa):")
print(diffuser_example.draw(output="text"))

Phase oracle, stato marcato |101>:
                    
q_0: ───────■───────
     ┌───┐  │  ┌───┐
q_1: ┤ X ├──■──┤ X ├
     ├───┤┌─┴─┐├───┤
q_2: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘
Diffusore (fase globale pi inclusa):
global phase: π
     ┌───┐┌───┐          ┌───┐┌───┐     
q_0: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_1: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐
q_2: ┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘└───┘└───┘└───┘└───┘


### 6.2 Verifica delle ampiezze e tabella
Prepariamo $|s\rangle$ con Hadamard e confrontiamo le ampiezze con formule indipendenti dai circuiti.
Per $N=8,M=1$, dopo l'oracolo $\bar a=3/(4\sqrt8)$; dopo il diffusore
l'ampiezza marcata vale $5/(2\sqrt8)$ e le altre $1/(2\sqrt8)$.
Gli assert usano tolleranza assoluta $10^{-12}$ e relativa zero.

In [6]:
ATOL = 1e-12
prepare_uniform = QuantumCircuit(3)
prepare_uniform.h(range(3))
before_oracle = Statevector.from_instruction(prepare_uniform)
after_oracle = before_oracle.evolve(oracle_example)
after_diffuser = after_oracle.evolve(diffuser_example)
expected_before = np.full(8, 1 / np.sqrt(8), dtype=complex)
expected_after = expected_before.copy()
expected_after[5] *= -1
expected_diffused = 2 * expected_after.mean() - expected_after
np.testing.assert_allclose(before_oracle.data, expected_before, atol=ATOL, rtol=0)
np.testing.assert_allclose(after_oracle.data, expected_after, atol=ATOL, rtol=0)
np.testing.assert_allclose(after_diffuser.data, expected_diffused, atol=ATOL, rtol=0)
np.testing.assert_allclose(after_diffuser.data[5], 5 / (2 * np.sqrt(8)), atol=ATOL, rtol=0)
np.testing.assert_allclose(np.linalg.norm(after_diffuser.data), 1, atol=ATOL, rtol=0)
amplitudes_df = pd.DataFrame({
    "basis_state": [format(i, "03b") for i in range(8)],
    "amplitude_before_oracle": np.real_if_close(before_oracle.data),
    "amplitude_after_oracle": np.real_if_close(after_oracle.data),
    "amplitude_after_diffuser": np.real_if_close(after_diffuser.data),
})
display(amplitudes_df)

,basis_state,amplitude_before_oracle,amplitude_after_oracle,amplitude_after_diffuser
0,000,0.353553,0.353553,0.176777
1,001,0.353553,0.353553,0.176777
2,010,0.353553,0.353553,0.176777
3,011,0.353553,0.353553,0.176777
4,100,0.353553,0.353553,0.176777
5,101,0.353553,-0.353553,0.883883
6,110,0.353553,0.353553,0.176777
7,111,0.353553,0.353553,0.176777


### 6.3 Controlli degli operatori e dei casi limite
Il confronto delle matrici complete verifica ogni stato di base e, per linearità, ogni sovrapposizione.
Controlliamo n=1, CZ, MCX, stati non palindromi (necessari per rilevare inversioni dei bit),
più soluzioni e tutti gli stati marcati. Verifichiamo anche vettori complessi, fase globale e input invalidi.
Questi sono test di correttezza su casi piccoli, non esperimenti di scaling.

In [7]:
checked_oracles = 0
test_rng = np.random.default_rng(RANDOM_SEED)
for n in range(1, 5):
    size = 2 ** n
    basis = [format(i, f"0{n}b") for i in range(size)]
    uniform = np.full(size, 1 / np.sqrt(size), dtype=complex)
    identity = np.eye(size)
    expected_d = 2 * np.outer(uniform, uniform.conj()) - identity
    diffuser = build_diffuser(n)
    actual_d = Operator(diffuser).data
    np.testing.assert_allclose(actual_d, expected_d, atol=ATOL, rtol=0)
    np.testing.assert_allclose(actual_d @ uniform, uniform, atol=ATOL, rtol=0)
    np.testing.assert_allclose(actual_d @ actual_d, identity, atol=ATOL, rtol=0)
    np.testing.assert_allclose(actual_d.conj().T, actual_d, atol=ATOL, rtol=0)
    mcz = QuantumCircuit(n)
    apply_multi_controlled_z(mcz)
    diagonal = np.ones(size)
    diagonal[-1] = -1
    np.testing.assert_allclose(Operator(mcz).data, np.diag(diagonal), atol=ATOL, rtol=0)
    amplitudes = test_rng.normal(size=size) + 1j * test_rng.normal(size=size)
    amplitudes /= np.linalg.norm(amplitudes)
    np.testing.assert_allclose(
        Statevector(amplitudes).evolve(diffuser).data,
        2 * amplitudes.mean() - amplitudes, atol=ATOL, rtol=0,
    )
    without_global_phase = diffuser.copy()
    without_global_phase.global_phase = 0
    np.testing.assert_allclose(Operator(without_global_phase).data, -expected_d, atol=ATOL, rtol=0)
    np.testing.assert_allclose(
        Statevector(amplitudes).evolve(without_global_phase).probabilities(),
        Statevector(amplitudes).evolve(diffuser).probabilities(), atol=ATOL, rtol=0,
    )
    cases = [[state] for state in basis] + [basis, [basis[0], basis[-1]]]
    if n == 3:
        cases.append(["001", "110"])
    for marked in cases:
        signs = np.ones(size)
        signs[[int(state, 2) for state in marked]] = -1
        oracle = build_phase_oracle(n, marked)
        np.testing.assert_allclose(Operator(oracle).data, np.diag(signs), atol=ATOL, rtol=0)
        state_after = Statevector(uniform).evolve(oracle)
        expected = uniform * signs
        np.testing.assert_allclose(state_after.data, expected, atol=ATOL, rtol=0)
        np.testing.assert_allclose(
            state_after.evolve(diffuser).data, 2 * expected.mean() - expected,
            atol=ATOL, rtol=0,
        )
        checked_oracles += 1

for invalid_n in (0, -1, True, 1.5):
    for builder in (lambda n: build_phase_oracle(n, ["0"]), build_diffuser):
        try:
            builder(invalid_n)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Accettato n non valido: {invalid_n!r}")
for invalid_marked in ([], ["101", "101"], ["10"], ["10a"], [101], "101", None):
    try:
        build_phase_oracle(3, invalid_marked)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Accettati stati non validi: {invalid_marked!r}")
try:
    apply_multi_controlled_z(QuantumCircuit(0))
except ValueError:
    pass
else:
    raise AssertionError("L'helper ha accettato un circuito senza qubit.")
np.testing.assert_allclose(
    Operator(build_oracle(3, ("001", "110"))).data,
    Operator(build_phase_oracle(3, ["001", "110"])).data, atol=ATOL, rtol=0,
)
print(f"Verifiche superate: {checked_oracles} oracoli, diffusori e helper per n=1..4; "
      "ampiezze, endianness, fase globale e validazione input corretti.")

Verifiche superate: 39 oracoli, diffusori e helper per n=1..4; ampiezze, endianness, fase globale e validazione input corretti.


## 7. Costruzione dell'algoritmo di Grover
Prepareremo la sovrapposizione uniforme e applicheremo oracolo e diffusore per un numero parametrico di iterazioni. Le misure saranno opzionali per consentire sia controlli di stato sia campionamento; anche zero iterazioni sarà un caso valido.

In [8]:
def build_grover_circuit(
    n: int,
    marked_states: tuple[str, ...],
    iterations: int,
    *,
    measure: bool = True,
) -> QuantumCircuit:
    """Comporrà preparazione uniforme e iterazioni manuali di Grover.

    iterations dovrà essere un intero non negativo; se measure=True,
    il qubit i sarà misurato nel bit classico i.
    """
    raise NotImplementedError("Circuito di Grover da implementare nella fase successiva.")

## 8. Verifica teorica
Confronteremo i risultati ideali con $P_k=\sin^2((2k+1)\theta)$, dove $\theta=\arcsin\sqrt{M/N}$ e $N=2^n$, per preparazione uniforme e oracolo di fase ideale. Valuteremo gli interi vicini al primo massimo continuo $k=\pi/(4\theta)-1/2$.

Questa previsione servirà a verificare i circuiti e a individuare l'overshooting: continuare oltre il primo massimo può ridurre la probabilità di successo. Il calcolo e il confronto saranno implementati nella prossima fase.

## 9. Esperimenti ideali
Studieremo prima un piccolo caso, includendo zero iterazioni, il primo massimo e valori oltre il massimo. Confronteremo probabilità teoriche e frequenze campionate, considerando l'incertezza dovuta agli shots. Nessun circuito sperimentale viene eseguito ora.

## 10. Esperimenti con rumore
Introdurremo un modello di rumore esplicito e documentato, mantenendo confrontabili problema, shots e compilazione. Studieremo come la profondità influisce sul successo; modello e intensità del rumore verranno registrati nei metadati dell'esperimento.

## 11. Analisi di scaling
Useremo i valori iniziali di n per confrontare probabilità, iterazioni e risorse del circuito. Genereremo stati marcati della lunghezza corretta per ogni n. Distingueremo la complessità in interrogazioni all'oracolo dal costo delle porte e dal tempo del simulatore classico.

## 12. Più soluzioni
Estenderemo gli esperimenti a insiemi di M stati distinti. Il successo sarà la somma delle probabilità di tutti gli stati marcati; confronteremo il primo massimo con la previsione dipendente da M/N.

## 13. Eventuale hardware IBM
Valuteremo in seguito accesso, disponibilità e costo dei backend. Aggiungeremo allora le dipendenze necessarie e registreremo backend, job ID, compilazione e calibrazione. Non inseriremo credenziali nel notebook e non inviamo job in questa fase.

## 14. Raccolta dati
Prepariamo una tabella vuota con tipi espliciti e una funzione che crea tabelle indipendenti. Ogni riga futura rappresenterà un'esecuzione; `marked_states` conterrà una tupla di stringhe e `simulation_type` distinguerà ideale, rumore e hardware.

`execution_time` sarà espresso in secondi, misurando l'intervallo dalla sottomissione alla disponibilità del risultato (inclusa l'eventuale coda hardware), escludendo costruzione e transpilation. Profondità e conteggi saranno misurati sul circuito transpiled: conteremo le istruzioni di porta, escludendo misure, barriere e reset; il conteggio a due qubit includerà solo porte su esattamente due qubit. Le impostazioni di compilazione andranno registrate per rendere confrontabili queste metriche.

In [9]:
RESULT_DTYPES = {
    "backend": "string",
    "simulation_type": "string",
    "n_qubits": "Int64",
    "N": "Int64",
    "M": "Int64",
    "marked_states": "object",  # Tupla immutabile di stringhe binarie.
    "grover_iterations": "Int64",
    "shots": "Int64",
    "success_probability": "Float64",
    "execution_time": "Float64",  # Secondi.
    "circuit_depth": "Int64",
    "gate_count": "Int64",
    "two_qubit_gate_count": "Int64",
}


def make_results_dataframe() -> pd.DataFrame:
    """Crea una tabella vuota; i tipi nullable ammettono metriche mancanti."""
    return pd.DataFrame({
        name: pd.Series(dtype=dtype) for name, dtype in RESULT_DTYPES.items()
    })


results_df = make_results_dataframe()
results_df

,backend,simulation_type,n_qubits,N,M,marked_states,grover_iterations,shots,success_probability,execution_time,circuit_depth,gate_count,two_qubit_gate_count


## 15. Grafici
Useremo matplotlib per probabilità rispetto alle iterazioni, confronto ideale/rumore e risorse rispetto a n. Aggiungeremo etichette, unità e intervalli di incertezza; evitiamo grafici finché non saranno disponibili dati reali.

## 16. Conclusioni sperimentali
In seguito riassumeremo ciò che i dati supportano su massimo di successo, overshooting, rumore e scaling, distinguendo osservazioni e limiti. Questa fase non produce ancora conclusioni sperimentali.

### Controlli minimi della struttura
Verifichiamo configurazione, schema e tipi della tabella, generatore casuale e creazione degli oggetti Qiskit e matplotlib. Non chiamiamo i segnaposto; le sole evoluzioni Statevector sono i controlli dei componenti nella sezione 6.

In [10]:
assert SHOTS_DEFAULT > 0
assert all(isinstance(n, int) and n > 0 for n in N_VALUES)
validate_problem(N_QUBITS_DEFAULT, MARKED_STATES_DEFAULT)
assert 1 <= N_SOLUTIONS_DEFAULT <= SEARCH_SPACE_SIZE
assert results_df.empty
assert list(results_df.columns) == list(RESULT_DTYPES)
assert {name: str(dtype) for name, dtype in results_df.dtypes.items()} == RESULT_DTYPES
assert make_results_dataframe() is not results_df
assert np.array_equal(
    np.random.default_rng(RANDOM_SEED).random(4),
    np.random.default_rng(RANDOM_SEED).random(4),
)
assert QuantumCircuit(N_QUBITS_DEFAULT).num_qubits == N_QUBITS_DEFAULT
assert make_ideal_backend().options.seed_simulator == RANDOM_SEED
fig = plt.figure()
plt.close(fig)
print("Controlli minimi superati: ambiente e strutture pronti; nessun esperimento eseguito.")

Controlli minimi superati: ambiente e strutture pronti; nessun esperimento eseguito.
